In [ ]:
'''
Here based on code 1 within the same folder, we used a new method suggested by Ricardo to generate the mutation effect. This new 
method is much more computational efficient.

New mutate method within the code.

'''

In [1]:
from __future__ import division
import numpy as np
from scipy import stats
import scipy.spatial as spa
import numpy.random as rnd
import copy
import time
import pandas as pd
import math
import pickle

In [2]:
class Population(object):
    '''Define a single population in FGM. Assuming the fixed optimum is at the orgin of the Coordinate System'''

    def __init__(self,size,dim,fit,mut_rate,sigma, ploidy =1):
        
        self.initial_fitness = fit  # The initial fitness of the population
        self.dim = dim  # The dimension of the landscape
        self.size = size # The population size 
        self.mutation_rate = mut_rate/(dim*ploidy) # mutation rate
        self.sigma = sigma  # The STD of generating new mutations
        
        self.ploidy = ploidy  # The ploidy of the individual within the population
        
        self.soma_mut_moving_pos = np.zeros((size, dim, ploidy)) # These two np.array will be used to store how much that each mutation in each locus in
        self.germ_mut_moving_pos = np.zeros((size, dim, 2))     # Soma and Germ will move along the corresponding dimension. Initially no mutations
                                                                # Thus the values within the array are 0.
        
        self.generate_ancestor()  # Generate ancestors with pre-defined initial fitness
        self.create_pop()  # Create populations composed of ancestors
        
        self.current_step =0  # Will be used to indicate the number of generations that have been simulated.



    
        
    def calculate_pop_fitness(self): 
        '''Calculate the fitness of each individual within the whole population simultaneously'''
        
        euclidean_distance = np.sqrt(np.sum((self.pop_point-np.zeros((self.size,self.dim)))**2, axis =1))      
        fitness = np.exp(-(euclidean_distance**2)/2)
        return fitness


    
    def generate_ancestor(self): 
        '''Generate ancestor with the predefined initial fitness'''
   
        if self.initial_fitness == 1: #special case if want ancestor at optimum
            self.anc_point = np.zeros(self.dim) #place ancestor at optimum
            self.anc_fit = 1
        else:
            self.anc_point = np.random.normal(size=self.dim,scale=.01) 
            self.anc_fit = np.exp(-(spa.distance.euclidean(self.anc_point,np.zeros(self.dim))**2)/2) 
            scaling = 1 


            if self.anc_fit < self.initial_fitness: 

                while self.anc_fit < self.initial_fitness: 
                    scaling = scaling + .0001 #increase scalar
                    self.anc_point = self.anc_point/scaling 
                    self.anc_fit = np.exp(-(spa.distance.euclidean(self.anc_point,np.zeros(self.dim))**2)/2) #test fitness

            elif self.anc_fit > self.initial_fitness: 

                while self.anc_fit > self.initial_fitness:  #keep moving ancestor away from optimum until desired initial fitness reached
                    scaling = scaling + .0001 #increase scalar
                    self.anc_point = self.anc_point * scaling 
                    self.anc_fit = np.exp(-(spa.distance.euclidean(self.anc_point,np.zeros(self.dim))**2)/2)            
            

            
    def create_pop(self):
        """ Create population - initially monomorphic. """

        self.anc_pop_point = np.repeat([self.anc_point], self.size, axis =0)
        
        self.pop_point = np.repeat([self.anc_point], self.size, axis =0)  
        self.pop_fit = np.repeat([float(self.anc_fit)], self.size, axis =0)
        
        self.soma_site = np.ones((self.size, self.dim, self.ploidy), dtype ='int')
        self.germ_site = np.ones((self.size, self.dim, 2), dtype ='int')

        
        
    def mutate(self):
        '''Generate new mutations in both Soma and Germ'''
        
        # Use the new method introduced by Ricardo.
        
        soma_mut_occur = np.random.binomial(self.soma_site, self.mutation_rate)
        germ_mut_occur = np.random.binomial(self.germ_site, self.mutation_rate)
        
        soma_mut_move = np.random.normal(0, self.sigma, size = self.size*self.dim*self.ploidy).reshape(self.size, self.dim, self.ploidy)
        germ_mut_move = np.random.normal(0, self.sigma, size = self.size*self.dim*2).reshape(self.size, self.dim, 2)
        
        soma_mut_movement = soma_mut_occur*soma_mut_move
        germ_mut_movement = germ_mut_occur*germ_mut_move
        
        soma_mut_moving_pos = self.soma_mut_moving_pos.copy()*(1-soma_mut_occur)
        germ_mut_moving_pos = self.germ_mut_moving_pos.copy()*(1-germ_mut_occur)
        
        self.soma_mut_moving_pos = soma_mut_moving_pos+soma_mut_movement
        self.germ_mut_moving_pos = germ_mut_moving_pos+germ_mut_movement
        
        self.pop_point = self.anc_pop_point + np.sum(self.soma_mut_moving_pos, axis =2)
        self.pop_fit = self.calculate_pop_fitness()        


       
    def selection(self):
        '''Select the parents for next generation based on their relative fitness. Select with replacement.'''

        assert self.pop_fit.min() >= 0 # ensure that the fitness of every one within the population is non-negative.
        
        total_w = np.sum(self.pop_fit)   # First caclulate the total fitness within the population
        relfit = self.pop_fit/total_w  # Calculate the relative fitness of each individual
        csrelfit = np.cumsum(relfit)
        randvals = np.random.random(self.size)
        parents = np.searchsorted(csrelfit,randvals)
        
        return parents
    
    
    def mitosis(self, parents):

        '''Asexual reproduction by mitosis. Just get an copy of the selected parents (inclduing the position, the fitness of the population, and
        the moving position of mutations in both Soma and Germ .'''

             
        self.soma_mut_moving_pos = self.soma_mut_moving_pos[parents]
        self.germ_mut_moving_pos = self.germ_mut_moving_pos[parents]
        
        # Then based on the new soma_moving_pos, recalculate the pop_point and pop_fit
        self.pop_point = self.anc_pop_point + np.sum(self.soma_mut_moving_pos, axis =2)
        self.pop_fit = self.calculate_pop_fitness()
      
    
    def amitosis(self, parents):
        '''Asexual reproduction by amitosis.'''
        
        self.soma_mut_moving_pos = self.soma_mut_moving_pos[parents] # First get the moving positions of mutations in Soma of selected parents
        new_soma_mut_moving_pos = np.repeat(self.soma_mut_moving_pos, 2, axis =2) # Duplicate the mutations in Soma of selected parents
        
        # Make sure that shuffle can actually do the shuffling. Maybe try random.choice first.
        for i in range(self.size):
            for j in range(self.dim):
                np.random.shuffle(new_soma_mut_moving_pos[i][j])  # Shuffle the duplicated mutations in Soma
                
        self.soma_mut_moving_pos = new_soma_mut_moving_pos[:, :, :self.ploidy]  # Pick the first self.ploidy mutations as the mutations in offspring Soma
        
        self.pop_point = self.anc_pop_point + np.sum(self.soma_mut_moving_pos, axis =2)  # Move the population to the new position.
        
        self.pop_fit = self.calculate_pop_fitness()      #  Calculate the fitness of new populations 
       
        self.germ_mut_moving_pos = self.germ_mut_moving_pos[parents]   # Get a copy of the Germ mutations in selected parents
        
        
        
    def asexual_reproduction(self,asex_type='amitosis'):
        """Take populations through a single Wright-Fisher model generation"""

        self.mutate()
        parents = self.selection()
        if asex_type == 'amitosis':
            self.amitosis(parents)
        elif asex_type == 'mitosis':
            self.mitosis(parents)
        self.current_step += 1

    

    def make_gametes(self, parents):
        """Generate gametes from the germ once having sex."""
        
        germ_mut_moving_pos = self.germ_mut_moving_pos[parents].copy()  # First get a copy of mutations in the Germ of selected parents
        
        for i in range(self.size):
            for j in range(self.dim):
                np.random.shuffle(germ_mut_moving_pos[i][j])  # Shuffle the mutations in selected Germ 
                
        gametes_mut_moving_pos = germ_mut_moving_pos[:, :, 0]  # Pick the first mutation in each locus (dimension) to generate gametes.
        
        return gametes_mut_moving_pos
    
    
    
    def make_zygotes(self, random_mating=False):
        """Generate zygotes from the gametes during sex"""

        if random_mating == False: 
            # Selfing. Select parents once, and the selected parents generate gametes twice.
            parents = self.selection()

            these_gamete_mut_moving_pos = self.make_gametes(parents)
            those_gamete_mut_moving_pos = self.make_gametes(parents)

        else: 
            # Random mating. Select parent twice, and each set of selected parents generates gametes once.
            first_parents = self.selection()
            second_parents = self.selection()

            these_gamete_mut_moving_pos = self.make_gametes(first_parents)
            those_gamete_mut_moving_pos = self.make_gametes(second_parents)           

        
        final_these_gamete_mut_moving_pos = np.expand_dims(these_gamete_mut_moving_pos, axis =2)
        final_those_gamete_mut_moving_pos = np.expand_dims(those_gamete_mut_moving_pos, axis =2)
        
        # Generate zygotes from the gametes. The zygote_mut_moving_pos stores the moving positions of each mutations in the new formed zygotes.
        zygote_mut_moving_pos = np.append(final_these_gamete_mut_moving_pos, final_those_gamete_mut_moving_pos, axis =2)
    
        return zygote_mut_moving_pos
    
    
    
    def sex(self,random_mating=False):
        '''Generate new Soma and Germ through a sexual process.'''
        
        self.mutate() # First mutate both Soma and Germ
        
        zygote = self.make_zygotes(random_mating)  # Generate zygotes by sex.
        
        self.germ_mut_moving_pos = zygote  # The new Germ is identical to zygote (i.e., the mutations in new Germ is same as that in zygotes).
        
        # Then from the zygote develop the new Soma.
        part_soma_mut_moving_pos = np.repeat(zygote,int(self.ploidy/2), axis=2 ) # First replicate each mutations in zygotes int(self.ploidy/2) times.
                                                                                # (similar to our original model, for heterozygous loci in zygotes, it 
                                                                                # will be 22 mutation + 22 WT + 1 MT or WT.)
        
        if self.ploidy - 2*int(self.ploidy/2) ==0:  # If self.ploidy is even, then part_soma_mut_moving_pos is directly the new Soma.
            self.soma_mut_moving_pos = part_soma_mut_moving_pos
            
        else:  # If self.ploidy is odd, then we need to get the remaining mutations for each locus (simuilar to the remaining 1 MT or WT for heterzygous
                # loci in our original model). Here we pick one from the two mutations in each locus of zygotes randomly.
            remain_soma = []
            remain_index = np.random.randint(0,2, size = (self.size, self.dim))  
            
            for i in range(self.size):
                ind_remain_soma = []
                for j in range(self.dim):
                    ind_remain_soma.append([zygote[i][j][remain_index[i][j]]])
                remain_soma.append(ind_remain_soma)
            remain_soma = np.array(remain_soma)  # Get the remaining mutations for each locus in the new Soma
            
            self.soma_mut_moving_pos = np.append(part_soma_mut_moving_pos, remain_soma, axis =2) # Append the remaining mutations for each locus into the new Soma.
            
        
        self.pop_point = self.anc_pop_point + np.sum(self.soma_mut_moving_pos, axis =2) # Move the population to the new position.
        
        self.pop_fit = self.calculate_pop_fitness()  # Calculate the fitness of new populations
        
        self.current_step += 1  # Update the generation




    def get_results(self):
        '''Get soma data from the simulation'''

        W = self.pop_fit

        W_mean = np.nanmean(W)  # Population mean fitness
        W_std = np.nanstd(W)   # STD of the fitness within the population.
        W_var = np.var(W)  # Variance of the fitness within the population.


        return [W_mean, W_std, W_var]



    def simulate(self,stepcount, asex_type='amitosis',sex_freq=None,random_mating=False,strides=10):

        '''Run the simulation for certain generations with defined reproduction strategy.'''
        
        """stepcount - total number of generations or time steps to run
        asex_type - type of asexual reproduction: mitosis, amitosis
        sex_freq - frequency of sexual reproduction
        random_mating - selfing or random mating
        strides - how frequently to get data using the get_results method
        
        returns results as a list of lists"""


        results = [self.get_results()]

        start = time.time()

        while self.current_step <= stepcount:
            if (sex_freq != None) and (self.current_step%sex_freq == 0):
                self.sex(random_mating)
            else:
                self.asexual_reproduction(asex_type)


            if self.current_step%strides == 0:
                results.append(self.get_results())


        colnames = ['meanFit','StdFit', 'VarFit']

        results = pd.DataFrame(np.array(results),columns=colnames)


        end = time.time()
        print 'TOTAL TIME: ',end-start
        return results



    def simulateNsave(self,outfile,stepcount,asex_type='amitosis',sex_freq=None,random_mating=False,strides=10):
        """same as simulate method except results are written to the file specified by outfile"""
        results = self.simulate(stepcount, asex_type,sex_freq,random_mating,strides)
        results.to_csv(outfile,index=False)  
        
        return